In [3]:
import pandas as pd
import numpy as np

# ===== CONFIG =====
PAYMENTS_FILE = "payments.csv"
OUT_FILE = "psp_settlement.csv"
C_TXN, C_DATE, C_TYPE, C_AMT, C_STATUS = "txn_id", "txn_time", "txn_type", "amount", "status"
SUCCESS_VALUES = {"completed", "success", "successful", "approved"}
DEP_FEE_RATE = 0.015   # 1.5% on deposits
WD_FEE_FLAT = 0.50     # flat fee on withdrawals
N_EXTRA = 15           # PSP rows that don't exist in your ledger
SEED = 42
# ==================

df = pd.read_csv(PAYMENTS_FILE)
df = df.drop_duplicates(subset=C_TXN).copy()

if C_STATUS in df.columns:
    print("Status values found:", df[C_STATUS].value_counts().to_dict())
    df = df[df[C_STATUS].astype(str).str.lower().str.strip().isin(SUCCESS_VALUES)].copy()

df[C_TYPE] = df[C_TYPE].astype(str).str.lower().str.strip()
df[C_AMT] = pd.to_numeric(df[C_AMT])
df[C_DATE] = pd.to_datetime(df[C_DATE])

rng = np.random.default_rng(SEED)
n = len(df)
r = rng.random(n)

# Disjoint slices of the random draw decide which rows get which problem
is_missing = r < 0.005                      # in ledger, absent from PSP
is_amt     = (r >= 0.005) & (r < 0.011)     # amount differs
is_fee     = (r >= 0.011) & (r < 0.016)     # fee differs
is_dup     = (r >= 0.016) & (r < 0.019)     # sent twice by PSP

# Fee is always calculated on the LEDGER amount
fee = np.where(df[C_TYPE] == "deposit", (df[C_AMT] * DEP_FEE_RATE).round(2), WD_FEE_FLAT)
psp_amt = df[C_AMT].to_numpy().copy()
psp_fee = fee.copy()

sign = rng.choice([-1, 1], size=n)
psp_amt = np.where(is_amt, (psp_amt + sign * rng.uniform(1, 25, n)).round(2), psp_amt)
psp_fee = np.where(is_fee, (psp_fee + rng.uniform(0.5, 5, n)).round(2), psp_fee)

psp = pd.DataFrame({
    "txn_id": df[C_TXN].values,
    "amount": psp_amt,
    "fee": psp_fee,
    "currency": "EUR",
    "settle_date": (df[C_DATE] + pd.Timedelta(days=1)).dt.strftime("%Y-%m-%d").values,
})

dups = psp[is_dup]
psp = psp[~is_missing]
psp = pd.concat([psp, dups], ignore_index=True)

# Transactions the PSP has that your ledger does not
dates = df[C_DATE].dt.strftime("%Y-%m-%d").to_numpy()
extra = pd.DataFrame({
    "txn_id": [f"EXT{i:06d}" for i in range(1, N_EXTRA + 1)],
    "amount": rng.uniform(20, 500, N_EXTRA).round(2),
    "fee": WD_FEE_FLAT,
    "currency": "EUR",
    "settle_date": rng.choice(dates, N_EXTRA),
})
psp = pd.concat([psp, extra], ignore_index=True)

psp = psp.sample(frac=1, random_state=SEED).sort_values("settle_date", kind="stable").reset_index(drop=True)
psp.insert(0, "psp_ref", [f"PSP{i:07d}" for i in range(1, len(psp) + 1)])
psp.to_csv(OUT_FILE, index=False)

print(f"\nWrote {OUT_FILE}: {len(psp)} rows")
print("Expected reconciliation results:")
print(f"  Missing at PSP   : {int(is_missing.sum())}")
print(f"  Missing in ledger: {N_EXTRA}")
print(f"  Amount mismatch  : {int(is_amt.sum())}")
print(f"  Fee mismatch     : {int(is_fee.sum())}")
print(f"  Duplicate at PSP : {int(is_dup.sum())}")

Status values found: {'success': 47971, 'failed': 2681}

Wrote psp_settlement.csv: 47850 rows
Expected reconciliation results:
  Missing at PSP   : 269
  Missing in ledger: 15
  Amount mismatch  : 286
  Fee mismatch     : 232
  Duplicate at PSP : 133
